# Unit 1 - Exercise 4: Data Preprocessing Practice

**GDI Theme**: Financial Investigations - Preparing transaction data for fraud detection models

**Dataset:** the real credit-card fraud dataset in this course's own data folder —
`../../datasets/raw/creditcard_fraud.csv`. It holds **284,807 genuine card transactions**
made by European cardholders over two days in September 2013, of which **492 were
confirmed frauds (0.17%)**. Most columns are `V1`–`V28`, the output of a PCA transform the
publishers applied to protect cardholders' privacy; `Time` and `Amount` are untouched.

## Instructions:
1. Load the real dataset provided below
2. Identify numerical and categorical features
3. Apply feature scaling (StandardScaler and MinMaxScaler)
4. Encode categorical variables (LabelEncoder and OneHotEncoder)
5. Split data into training and testing sets
6. Compare different preprocessing methods
7. Create visualizations comparing before/after scaling

⚠️ **Look at the class balance before you do anything else.** Fraud is well under 1% of
rows (the cell below prints the exact rate for the slice you load), so a model that
predicts "not fraud" every single time scores over 99% accuracy and is completely
useless. Real fraud data teaches that on the first line; a balanced made-up dataset hides it.

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# Import the preprocessing toolkit.
# WHY: scalers put features on comparable ranges; encoders turn categories
# into numbers - both are needed before most models can train.
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OneHotEncoder
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Task 1: Generate Sample Dataset

In [2]:
# Load the REAL credit-card transaction data that ships with this course.
# WHY real data: the class balance (0.17% fraud), the enormous spread of transaction
# amounts, and the fact that most columns are already PCA-transformed are all facts
# about this dataset that shape how you must preprocess it.
FRAUD_CSV = "../../datasets/raw/creditcard_fraud.csv"

# Classroom-size the file: read a subset of columns and a slice of rows so the notebook
# stays fast. We keep Time and Amount (the two untransformed real columns) plus four of
# the anonymised PCA components, and the real fraud label.
df = pd.read_csv(
    FRAUD_CSV,
    usecols=["Time", "Amount", "V1", "V2", "V3", "V4", "Class"],
    nrows=50_000,        # the first 50,000 real transactions, in time order
)

# --- Feature engineering on REAL columns (not invented data) ---
# 'Time' is seconds elapsed since the first transaction in the file. Turn it into the
# real clock hour, then bucket it: a NOMINAL category for one-hot encoding.
df["hour_of_day"] = ((df["Time"] // 3600) % 24).astype(int)
df["time_of_day"] = pd.cut(
    df["hour_of_day"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"],
).astype(str)

# 'Amount' is the real transaction value in euros. Split it at real quantiles into an
# ORDINAL category — Low < Medium < High — for label encoding.
df["amount_band"] = pd.qcut(df["Amount"], q=3, labels=["Low", "Medium", "High"]).astype(str)

# A readable name for the real target column.
df["is_fraud"] = df["Class"]
df = df.drop(columns=["Class", "Time"])

print("✅ Real dataset loaded!")
print(f"   📊 Shape: {df.shape}")
print(f"   📊 Columns: {list(df.columns)}")
print("\n📋 Data Types:")
print(df.dtypes)
print("\n📋 First few rows:")
print(df.head())
print("\n⚠️  Real class balance:")
print(df["is_fraud"].value_counts())
print(f"   Fraud rate in this slice: {df['is_fraud'].mean():.4%}")
print("\n📋 Real transaction amounts (note the range — this is why scaling matters):")
print(df["Amount"].describe().round(2))

✅ Real dataset loaded!
   📊 Shape: (50000, 9)
   📊 Columns: ['V1', 'V2', 'V3', 'V4', 'Amount', 'hour_of_day', 'time_of_day', 'amount_band', 'is_fraud']

📋 Data Types:
V1             float64
V2             float64
V3             float64
V4             float64
Amount         float64
hour_of_day      int64
time_of_day     object
amount_band     object
is_fraud         int64
dtype: object

📋 First few rows:
         V1        V2        V3        V4  Amount  hour_of_day time_of_day  \
0 -1.359807 -0.072781  2.536347  1.378155  149.62            0       Night   
1  1.191857  0.266151  0.166480  0.448154    2.69            0       Night   
2 -1.358354 -1.340163  1.773209  0.379780  378.66            0       Night   
3 -0.966272 -0.185226  1.792993 -0.863291  123.50            0       Night   
4 -1.158233  0.877737  1.548718  0.403034   69.99            0       Night   

  amount_band  is_fraud  
0        High         0  
1         Low         0  
2        High         0  
3        High       

## Task 2: Identify Feature Types

TODO: Separate numerical and categorical features
Hint: Use df.select_dtypes(include=['number']) for numerical
Hint: Use df.select_dtypes(include=['object']) for categorical

Your code here...

## Task 3: Feature Scaling - StandardScaler

TODO: Apply StandardScaler to numerical features
Steps:
1. Select numerical features
2. Create StandardScaler object
3. Fit and transform the features
4. Display before/after statistics (mean, std)

Your code here...

## Task 4: Feature Scaling - MinMaxScaler

TODO: Apply MinMaxScaler to numerical features
Steps:
1. Create MinMaxScaler object
2. Fit and transform numerical features
3. Display before/after statistics (min, max)
4. Compare with StandardScaler results

Your code here...

## Task 5: Categorical Encoding - LabelEncoder

TODO: Apply LabelEncoder to ordinal categorical features
Steps:
1. Select the ordinal categorical feature (`amount_band`: Low < Medium < High)
2. Create LabelEncoder object
3. Fit and transform the feature
4. Display mapping (original values -> encoded values)

Your code here...

## Task 6: Categorical Encoding - OneHotEncoder

TODO: Apply OneHotEncoder to nominal categorical features
Steps:
1. Select the nominal categorical feature (`time_of_day`: Night / Morning / Afternoon / Evening)
2. Create OneHotEncoder object
3. Fit and transform the features
4. Convert to DataFrame for better visualization

Your code here...

## Task 7: Train-Test Split

TODO: Split data into training and testing sets
Steps:
1. Prepare features (X) - all columns except target
2. Prepare target (y) - `is_fraud`
3. Use train_test_split with test_size=0.2, random_state=73, **and `stratify=y`**
   (without stratify, a test split of this imbalanced real data may contain almost
   no fraud cases at all)
4. Display shapes of train/test sets

Your code here...

## Task 8: Complete Preprocessing Pipeline

TODO: Create a complete preprocessing pipeline
Steps:
1. Split data first (train/test)
2. Scale numerical features on training data
3. Apply scaling to test data (using scaler fitted on train)
4. Encode categorical features
5. Combine scaled numerical + encoded categorical features

Your code here...

## Task 9: Visualization - Compare Before/After Scaling

TODO: Create visualizations comparing original vs scaled features
Steps:
1. Plot distribution of original numerical feature
2. Plot distribution after StandardScaler
3. Plot distribution after MinMaxScaler
4. Use subplots to show side-by-side comparison

Your code here...

## ✅ Exercise Complete!

**What You Learned:**
- ✅ Feature scaling (StandardScaler vs MinMaxScaler)
- ✅ Categorical encoding (LabelEncoder vs OneHotEncoder)
- ✅ Train-test split for proper evaluation
- ✅ Complete preprocessing pipeline
- ✅ GDI context: Preparing real financial data for fraud detection
- ✅ Why severe real class imbalance changes how you split and evaluate